# Smoke / fire / normal — Colab training (TensorFlow only, no tensorflowjs)

**Goal:** Train with Colab's preinstalled GPU TensorFlow, save `fire_smoke.keras`, and download.

**Why not TF.js in Colab:** `tensorflowjs` conflicts with Colab Python 3.12, `pkg_resources`, and dependency resolution. Convert **locally**:
`python scripts/export_keras_to_tfjs.py ...\fire_smoke.keras tfjs-web-app\public\model` (needs `requirements-train.txt`).

**Suggested flow:**
1. **Runtime → Change runtime type → GPU**.
2. Section 2: check TensorFlow / GPU.
3. **Data:** prefer **section 3A Kaggle** (`amerzishminha/forest-fire-smoke-and-non-fire-image-dataset`, ~**7GB**, often **15–40 min**); or **section 3B Drive**.
4. **Train** → **zip `artifacts.zip`** → convert TF.js on your PC.

Local alternative: `python scripts/train_fire_smoke_keras.py`.

## 1) GPU (optional)
**Runtime → Change runtime type → T4 GPU** → Save. You can skip `nvidia-smi`.

In [ ]:
!nvidia-smi -L 2>/dev/null || true

## 2) Dependencies (Colab already has TensorFlow)

Do **not** `pip install tensorflowjs` in this notebook.

In [ ]:
# !pip install -q h5py  # only if a training cell reports a missing package

import tensorflow as tf
print("TensorFlow:", tf.__version__, "GPU:", tf.config.list_physical_devices("GPU"))

## 3) Data (pick one)

- **3A:** On Kaggle, accept rules / download if required, then run the next two cells (~**7GB**, downloaded by Colab).
- **3B:** `dataset.zip` on Google Drive.

After **3A** works, **skip 3B**.

### 3A) Kaggle: `amerzishminha/forest-fire-smoke-and-non-fire-image-dataset`

1. Accept dataset rules on the Kaggle page if prompted.
2. **Credentials (pick one; new token recommended):**
   - **New:** Kaggle **Settings → Account → API → Generate New Token**, copy **`KGAT_...`** (shown once). In Colab **Secrets**, add **`KAGGLE_API_TOKEN`**, paste the token, enable **Notebook access**.
   - **Legacy:** `kaggle.json` → Secrets **`KAGGLE_USERNAME`** + **`KAGGLE_KEY`**.
3. Run the download cell (~7GB; slow is normal).
4. Run the "layout to /content/dataset" cell.

In [ ]:
import json
import os

!pip install -q kagglehub

from google.colab import userdata


def _sec(name):
    try:
        v = userdata.get(name)
        return v if v else None
    except Exception:
        return None


api_tok = _sec("KAGGLE_API_TOKEN")
if api_tok:
    os.environ["KAGGLE_API_TOKEN"] = api_tok.strip()
    print("Using Secrets: KAGGLE_API_TOKEN (KGAT_...)")
else:
    u, k = _sec("KAGGLE_USERNAME"), _sec("KAGGLE_KEY")
    if not u or not k:
        raise RuntimeError(
            "Set Colab Secrets: KAGGLE_API_TOKEN (new), or KAGGLE_USERNAME + KAGGLE_KEY (legacy)"
        )
    os.makedirs("/root/.kaggle", exist_ok=True)
    with open("/root/.kaggle/kaggle.json", "w") as f:
        json.dump({"username": u, "key": k}, f)
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
    print("Using Secrets: KAGGLE_USERNAME + KAGGLE_KEY (legacy kaggle.json)")

import kagglehub

DATASET_SLUG = "amerzishminha/forest-fire-smoke-and-non-fire-image-dataset"
kaggle_path = kagglehub.dataset_download(DATASET_SLUG)
print("Downloaded and extracted to:", kaggle_path)

In [ ]:
import os
import shutil
from pathlib import Path


def folder_to_normal_smoke_fire(name: str):
    n = name.strip().lower().replace("_", " ")
    if n in ("non fire", "nonfire", "no fire"):
        return "normal"
    if n == "smoke":
        return "smoke"
    if n == "fire":
        return "fire"
    n2 = name.strip().lower().replace(" ", "-")
    if n2 == "non-fire":
        return "normal"
    return None


def find_train_test(base: Path):
    candidates = [base]
    if base.is_dir():
        candidates += [p for p in base.iterdir() if p.is_dir()]
    for root in candidates:
        tr = root / "train"
        te = root / "test"
        if tr.is_dir() and te.is_dir():
            return tr, te
    for tr in base.rglob("train"):
        if not tr.is_dir():
            continue
        te = tr.parent / "test"
        if te.is_dir():
            return tr, te
    return None, None


def copy_split(src_split: Path, dst_split: Path):
    dst_split.mkdir(parents=True, exist_ok=True)
    for sub in sorted(src_split.iterdir()):
        if not sub.is_dir():
            continue
        out = folder_to_normal_smoke_fire(sub.name)
        if out is None:
            print("Skipping unknown folder:", sub.name)
            continue
        out_dir = dst_split / out
        out_dir.mkdir(parents=True, exist_ok=True)
        for f in sub.iterdir():
            if f.is_file():
                shutil.copy2(f, out_dir / f.name)


base = Path(kaggle_path)
train_src, test_src = find_train_test(base)
assert train_src and test_src, f"train/test not found under base={base}"

DEST = Path("/content/dataset")
if DEST.exists():
    shutil.rmtree(DEST)
copy_split(train_src, DEST / "train")
copy_split(test_src, DEST / "val")

assert (DEST / "train" / "normal").is_dir()
assert (DEST / "val" / "fire").is_dir()
print("OK:", DEST, "sample train/normal count:", len(list((DEST / "train" / "normal").iterdir())))

### 3B) Or: `dataset.zip` on Google Drive

**If 3A succeeded, do not run the cells below.**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, zipfile, shutil

ZIP_ON_DRIVE = "/content/drive/MyDrive/ColabData/dataset.zip"

DEST = "/content/dataset"
if os.path.isdir(DEST):
    shutil.rmtree(DEST)
os.makedirs(DEST, exist_ok=True)
with zipfile.ZipFile(ZIP_ON_DRIVE, "r") as z:
    z.extractall(DEST)
for name in os.listdir(DEST):
    sub = os.path.join(DEST, name)
    if os.path.isdir(sub) and os.path.isdir(os.path.join(sub, "train")):
        for x in ["train", "val"]:
            shutil.move(os.path.join(sub, x), os.path.join(DEST, x))
        shutil.rmtree(sub)
        break
assert os.path.isdir("/content/dataset/train/normal")
print("OK:", DEST)

## 4) Train
Same as `scripts/train_fire_smoke_keras.py`. Try `EPOCHS=2` for a quick run.

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

CLASS_NAMES = ["normal", "smoke", "fire"]
BASE = "/content/dataset"
train_dir = f"{BASE}/train"
val_dir = f"{BASE}/val"
EPOCHS = 8
BATCH_SIZE = 32


def preprocess_tfjs_style(image):
    return image.astype(np.float32) / 127.0 - 1.0


def val_ready():
    for n in CLASS_NAMES:
        d = f"{val_dir}/{n}"
        if not os.path.isdir(d) or not os.listdir(d):
            return False
    return True


use_split = not val_ready()
print("Using 20% of train for validation" if use_split else "Using separate val/ folders")

train_aug = keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=preprocess_tfjs_style,
    horizontal_flip=True,
    zoom_range=0.1,
    brightness_range=(0.85, 1.15),
    validation_split=0.2 if use_split else 0.0,
)
train_flow = train_aug.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=CLASS_NAMES,
    shuffle=True,
    subset="training" if use_split else None,
)
if use_split:
    val_flow = train_aug.flow_from_directory(
        train_dir,
        target_size=(224, 224),
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        classes=CLASS_NAMES,
        shuffle=False,
        subset="validation",
    )
else:
    val_plain = keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=preprocess_tfjs_style,
    )
    val_flow = val_plain.flow_from_directory(
        val_dir,
        target_size=(224, 224),
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        classes=CLASS_NAMES,
        shuffle=False,
    )

base = keras.applications.MobileNetV2(
    include_top=False, weights="imagenet", input_shape=(224, 224, 3), pooling="avg"
)
base.trainable = False
inputs = base.input
out = layers.Dense(3, activation="softmax", name="probs")(base.output)
model = keras.Model(inputs, out, name="fire_smoke_mobilenet")
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
cb = keras.callbacks.EarlyStopping(
    patience=4, restore_best_weights=True, monitor="val_accuracy"
)
model.fit(train_flow, validation_data=val_flow, epochs=EPOCHS, callbacks=[cb])

model.save("/content/fire_smoke.keras")
print("Saved /content/fire_smoke.keras")

## 5) Download bundle

On your PC (project root):
`pip install -r requirements-train.txt`  
`python scripts/export_keras_to_tfjs.py <path>\fire_smoke.keras tfjs-web-app\public\model`

In [ ]:
!cd /content && zip -j artifacts.zip fire_smoke.keras && ls -la artifacts.zip